<div dir="rtl" style="text-align:right">
<h1>شکل درست، محور درست؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه خطای معنایی محور را حتی وقتی کد اجرا می‌شود پیدا کنیم؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-03/chapter-01/13-torch.html"><bdi dir="ltr">13-torch</bdi></a>، <a href="http://127.0.0.1:8000/part-03/chapter-01/14-index-device.html"><bdi dir="ltr">14-index-device</bdi></a>، <a href="http://127.0.0.1:8000/part-03/chapter-02/15-broadcast.html"><bdi dir="ltr">15-broadcast</bdi></a>، <a href="http://127.0.0.1:8000/part-03/chapter-02/16-reshape.html"><bdi dir="ltr">16-reshape</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای تکرار پاک، Kernel را Restart و سپس Run All کنید. لینک درس با سروکردن کتاب روی پورت ۸۰۰۰ کار می‌کند؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این نخستین دفتر PyTorch است. B تعداد نمونه، T تعداد موقعیت و C تعداد ویژگی است؛ نام‌ها را کنار Shape بنویسید. در مثال نخست فقط سطر و ستون داریم. پیش از اجرا شکل ضرب عضو‌به‌عضو و ضرب ماتریسی را جدا پیش‌بینی کنید.</p>
</div>

In [ ]:
x = torch.tensor([[1.,2.,3.],[4.,5.,6.]])
inspect("x", x)
print("Elementwise:", x*x)
print("Matrix product:", x @ x.T)
print("reshape(3,2):", x.reshape(3,2))
print("transpose:", x.T)
assert not torch.equal(x.reshape(3,2), x.T)
assert torch.equal(x @ x.T, torch.tensor([[14.,32.],[32.,77.]]))


<div dir="rtl" style="text-align:right">
<h2>شکست آشکار و شکست خاموش</h2><p style="text-align:right">می‌خواهیم میانگین هر سطر را از همان سطر کم کنیم. چرا حذف keepdim در جدول دو‌در‌سه خطا می‌دهد، اما در جدول سه‌در‌سه ممکن است اجرا شود و غلط باشد؟</p>
</div>

In [ ]:
try:
    x - x.mean(dim=1)
except RuntimeError as error:
    print("Expected broadcasting failure:", error)
else:
    raise AssertionError("Expected shape mismatch")
square = torch.arange(1.,10.).reshape(3,3)
wrong = square - square.mean(dim=1)
correct = square - square.mean(dim=1, keepdim=True)
inspect("row means with keepdim", square.mean(1, keepdim=True))
print("Wrong row means:", wrong.mean(1))
print("Correct row means:", correct.mean(1))
torch.testing.assert_close(wrong.mean(1), torch.tensor([-3.,0.,3.]))
torch.testing.assert_close(correct.mean(1), torch.zeros(3))


<div dir="rtl" style="text-align:right">
<h2>محور Batch را نگه داریم</h2><p style="text-align:right">قبل از اجرا شکل نتیجهٔ انتخاب موقعیت آخر و انتخاب یک نمونه را حدس بزنید. T را از ۴ به ۸ ببرید و فقط سطر تنظیمات را تغییر دهید.</p>
</div>

In [ ]:
B, T, C = 2, 4, 3
batch = torch.arange(B*T*C, dtype=torch.float32).reshape(B,T,C)
for name, value in [("batch",batch),("last position",batch[:,-1,:]),
                    ("one sample, keep B",batch[0:1]),("one sample, remove B",batch[0])]:
    inspect(name, value)
ids = torch.tensor([[0,1]], dtype=torch.long)
inspect("integer token IDs", ids)


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>برداشت:</b> Shape درست شرط لازم است، نه کافی. با یک مثال عددی توضیح دهید کدام محور در Broadcasting هم‌تراز می‌شود؛ فقط به نبودن پیام خطا اعتماد نکنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی، مشاهده و دلیل اختلافشان را در یادداشت خود بنویسید. سپس به <a href="http://127.0.0.1:8000/part-03/chapter-02/16-reshape.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>